# PDSCNN — Standalone Training (50 Epochs)

This notebook trains **only the PDSCNN branch** (no ViT, no RRELM, no fusion) from scratch, targeting the highest achievable standalone accuracy on the 4-class brain tumor MRI task (`glioma`, `meningioma`, `notumor`, `pituitary`).

Because the data is pooled from **multiple sources**, the preprocessing pipeline is built to remove cross-source inconsistency before the model ever sees the data:

| Stage | Purpose |
|---|---|
| 1. Source-aware loading | Merges `data.zip`, `data1.zip`, and `data2.zip` class-wise, tagging every image with which zip it came from |
| 2. Deduplication (perceptual hashing) | Removes exact/near-duplicate images that different sources often share, which would otherwise leak across train/val/test |
| 2a. Quality gate | Drops corrupt, unreadably small, blurry, or blank/low-information images before anything is selected for training |
| 2b. Strict fixed-size selection | From the deduped, quality-passed pool, keeps exactly the **12,500** most-unique images (class-balance preserving) |
| 3. Grayscale standardization | MRI scans should carry no color information; forces every source onto the same footing |
| 4. Brain-region autocrop | Strips black borders / scanner annotations that differ by source |
| 5. Denoising | Reduces scanner-specific sensor noise |
| 6. CLAHE | Normalizes contrast differences between scanners/sources |
| 7. Resize to 224×224 | Uniform input size |
| 8. Dataset-specific normalization | Mean/std computed on *this* processed dataset (not ImageNet, since PDSCNN trains from scratch) |
| 9. Augmentation (train only) | Flips, rotation, translation, zoom, brightness/contrast jitter |

Every preprocessing stage below shows before/after images so you can sanity-check it on your own data before committing to a 50-epoch run.

**Outputs saved to Google Drive:** best checkpoint, final checkpoint, full config (class map + normalization stats), training curves, confusion matrix, and the processed image cache.


## 0. Setup

In [1]:
!pip install -q imagehash grad-cam >/dev/null
print("Installed extra deps: imagehash, grad-cam")


In [2]:
import os, glob, shutil, random, zipfile, json, time, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, classification_report)

import imagehash

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)
if device.type == 'cuda':
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True


## 1. Load Data (multi-source aware)

Three known zips are merged **class-wise**: `/content/drive/MyDrive/data.zip`, `/content/drive/MyDrive/data1.zip`, and `/content/drive/MyDrive/data2.zip`. Each keeps its own `source` tag (`data`, `data1`, `data2`) so later steps can still check per-source distribution and dedup across them — but for training, class folders from all three zips are pooled together (e.g. every `glioma` folder from any zip becomes one `glioma` class).

A `data_sources/` folder with any number of additional zips is also still supported if you add more sources later. The code below auto-detects whichever of these are present.

**Note on nested folders:** if a zip has its own `Testing/`/`Training/` split before the class folders (e.g. `data2.zip/Training/Glioma/...` and `data2.zip/Testing/Glioma/...`), that's fine — class discovery below recurses through any folder depth and matches purely by folder name, so both get pooled into one `glioma` class. The original split is intentionally not preserved, since this notebook creates its own stratified train/val/test split later across the full merged, deduped, quality-filtered, size-selected pool.


In [ ]:
from google.colab import drive
import os, glob, shutil
import pandas as pd

drive.mount('/content/drive')

DRIVE_PROCESSED_ROOT = '/content/drive/MyDrive/processed_data_vit'
LOCAL_PROCESSED_ROOT = '/content/processed_data_vit'

if not os.path.exists(LOCAL_PROCESSED_ROOT):
    print("Copying preprocessed images from Drive to local SSD (faster I/O during training)...")
    shutil.copytree(DRIVE_PROCESSED_ROOT, LOCAL_PROCESSED_ROOT)
    print("Copy complete!")
else:
    print("Local cache already exists, skipping copy.")

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

records = []
for cls in CLASS_NAMES:
    cls_path = os.path.join(LOCAL_PROCESSED_ROOT, cls)
    if os.path.isdir(cls_path):
        for fname in os.listdir(cls_path):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                records.append({'processed_path': os.path.join(cls_path, fname), 'label': cls})

df = pd.DataFrame(records)
print(f"Loaded {len(df)} preprocessed images from local disk.")
print(df['label'].value_counts())

if len(df) == 0:
    raise ValueError('No images found! Make sure they are saved to ' + DRIVE_PROCESSED_ROOT)


## 4. Dataset-specific normalization stats

PDSCNN is trained **from scratch** here (not fine-tuned from an ImageNet checkpoint), so normalizing with ImageNet's mean/std is a mismatch — we compute mean/std directly from our own preprocessed training data instead.


In [ ]:
# Dataset-specific normalization stats (pre-computed from the processed dataset).
# Hardcoded to avoid re-scanning thousands of files on every run.
# If you change the preprocessing pipeline, delete these lines and uncomment the
# compute_mean_std() block below to recalculate.
DATASET_MEAN = [0.1919, 0.1919, 0.1919]
DATASET_STD  = [0.2058, 0.2058, 0.2058]
print("Using dataset mean:", DATASET_MEAN)
print("Using dataset std :", DATASET_STD)

# --- Uncomment below to recompute from scratch ---
# def compute_mean_std(paths, sample_size=2000):
#     if len(paths) > sample_size:
#         paths = random.sample(list(paths), sample_size)
#     pixel_sum = np.zeros(3)
#     pixel_sq_sum = np.zeros(3)
#     n_pixels = 0
#     for p in tqdm(paths, desc="Computing mean/std"):
#         arr = np.asarray(Image.open(p).convert('RGB'), dtype=np.float64) / 255.0
#         pixel_sum += arr.sum(axis=(0, 1))
#         pixel_sq_sum += (arr ** 2).sum(axis=(0, 1))
#         n_pixels += arr.shape[0] * arr.shape[1]
#     mean = pixel_sum / n_pixels
#     std = np.sqrt(pixel_sq_sum / n_pixels - mean ** 2)
#     return mean.tolist(), std.tolist()
# DATASET_MEAN, DATASET_STD = compute_mean_std(df['processed_path'].tolist())
# print("Dataset mean:", DATASET_MEAN)
# print("Dataset std :", DATASET_STD)


## 5. Train / Val / Test Split

Stratified by class on the deduplicated, preprocessed dataframe (70/15/15).


In [19]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))
print("\nTrain class balance:\n", train_df['label'].value_counts(normalize=True).round(3))


## 6. Augmentation & Dataset/DataLoader

Since heavy preprocessing is already cached, only fast, on-the-fly augmentation happens per epoch: horizontal flip, small rotation, translation/zoom, and mild brightness/contrast jitter. Only the training split is augmented; val/test just get resized + normalized.


In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32   # T4 has 15 GB VRAM; 32 is safe with PDSCNN + MixUp double pass avoided

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.90, 1.10), shear=5),
    transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(DATASET_MEAN, DATASET_STD),
    transforms.RandomErasing(p=0.20, scale=(0.02, 0.10)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(DATASET_MEAN, DATASET_STD),
])

# TTA (test-time augmentation) transform for evaluation
tta_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(DATASET_MEAN, DATASET_STD),
])

class BrainTumorDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['processed_path']).convert('RGB')
        label = CLASS_TO_IDX[row['label']]
        if self.transform:
            img = self.transform(img)
        return img, label

train_ds = BrainTumorDataset(train_df, train_transform)
val_ds   = BrainTumorDataset(val_df,   eval_transform)
test_ds  = BrainTumorDataset(test_df,  eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True,
                          prefetch_factor=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2)

print("Batches — train:", len(train_loader), "| val:", len(val_loader), "| test:", len(test_loader))


In [32]:
# Visualize a batch of augmented training images
def denorm(img_tensor):
    img = img_tensor.clone().numpy().transpose(1, 2, 0)
    img = img * np.array(DATASET_STD) + np.array(DATASET_MEAN)
    return np.clip(img, 0, 1)

imgs, labels = next(iter(train_loader))
fig, axs = plt.subplots(2, 6, figsize=(16, 6))
for i, ax in enumerate(axs.flatten()):
    if i >= len(imgs):
        ax.axis('off'); continue
    ax.imshow(denorm(imgs[i]))
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=9)
    ax.axis('off')
plt.suptitle("Augmented training samples")
plt.tight_layout()
plt.show()


## 7. PDSCNN Model

Same parallel depthwise-separable-conv architecture used in the hybrid pipeline (kept identical so this checkpoint can later be dropped straight into the fusion notebook), with a dropout head added for extra regularization on a from-scratch, standalone training run.


In [39]:
class DepthwiseSeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, padding=0):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=kernel_size,
                                   padding=padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        # Note: BN is applied after the pointwise conv in the sequential block in our wrapper,
        # but for the parallel branches we just return the raw conv output to concatenate
        return x

class PDSCNN(nn.Module):
    def __init__(self, num_classes=4, in_ch=3):
        super().__init__()

        # Parallel stage (Layer 1-5)
        # Branches have kernels 11, 9, 7, 5, 3 and padding to maintain SAME dimension (124)
        self.branch_11 = DepthwiseSeparableConv2d(in_ch, 64, 11, padding=5)
        self.branch_9  = DepthwiseSeparableConv2d(in_ch, 64, 9, padding=4)
        self.branch_7  = DepthwiseSeparableConv2d(in_ch, 64, 7, padding=3)
        self.branch_5  = DepthwiseSeparableConv2d(in_ch, 64, 5, padding=2)
        self.branch_3  = DepthwiseSeparableConv2d(in_ch, 64, 3, padding=1)

        # Sequential portion
        # Layer 6
        self.conv6 = DepthwiseSeparableConv2d(64 * 5, 256, 3, padding=0)
        self.bn6 = nn.BatchNorm2d(256)
        self.pool6 = nn.MaxPool2d(2, 2)

        # Layer 7
        self.conv7 = DepthwiseSeparableConv2d(256, 128, 3, padding=0)
        self.bn7 = nn.BatchNorm2d(128)
        self.pool7 = nn.MaxPool2d(2, 2)

        # Layer 8
        self.conv8 = DepthwiseSeparableConv2d(128, 64, 3, padding=0)
        self.bn8 = nn.BatchNorm2d(64)
        self.pool8 = nn.MaxPool2d(2, 2)

        # Layer 9
        self.conv9 = DepthwiseSeparableConv2d(64, 32, 3, padding=0)
        self.bn9 = nn.BatchNorm2d(32)
        self.pool9 = nn.MaxPool2d(2, 2)

        self.relu = nn.ReLU(inplace=True)
        self.flatten = nn.Flatten()

        # FC
        self.fc1 = nn.Linear(32, 512)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, 256)

        # Classifier (to support original notebook's training loop)
        self.classifier = nn.Linear(256, num_classes)

    def forward_features(self, x):
        # Parallel stage
        b11 = self.branch_11(x)
        b9  = self.branch_9(x)
        b7  = self.branch_7(x)
        b5  = self.branch_5(x)
        b3  = self.branch_3(x)

        x = torch.cat([b11, b9, b7, b5, b3], dim=1)

        x = self.pool6(self.bn6(self.relu(self.conv6(x))))
        x = self.pool7(self.bn7(self.relu(self.conv7(x))))
        x = self.pool8(self.bn8(self.relu(self.conv8(x))))
        x = self.pool9(self.bn9(self.relu(self.conv9(x))))
        x = nn.AdaptiveAvgPool2d(1)(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def forward(self, x):
        feat = self.forward_features(x)
        out = self.classifier(self.dropout(feat))
        return out, feat

pdscnn = PDSCNN(num_classes=len(CLASS_NAMES)).to(device)
n_params = sum(p.numel() for p in pdscnn.parameters())
print(f"PDSCNN parameters: {n_params:,}")

## 8. Training (50 epochs)

- **AdamW** + **cosine annealing** LR schedule with a short linear warmup
- **Label smoothing** cross-entropy (reduces overconfidence, helps generalization on a from-scratch model)
- **Class weighting** if the (deduplicated) split is imbalanced
- **Mixed precision (AMP)** for faster training on the Colab GPU
- **Gradient clipping** for stability
- Best checkpoint (by val accuracy) saved every time it improves; final checkpoint saved at the end


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import torch.optim.swa_utils as swa_utils

y_train_idx = train_df['label'].map(CLASS_TO_IDX).values
class_weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=y_train_idx)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", dict(zip(CLASS_NAMES, class_weights.round(3))))

# ---- MixUp helper ----
def mixup_data(x, y, alpha=0.4):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


In [ ]:
DRIVE_SAVE_DIR = '/content/drive/MyDrive/pdscnn_standalone_v2'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

BEST_CKPT_PATH  = os.path.join(DRIVE_SAVE_DIR, 'pdscnn_best.pth')
FINAL_CKPT_PATH = os.path.join(DRIVE_SAVE_DIR, 'pdscnn_final.pth')
SWA_CKPT_PATH   = os.path.join(DRIVE_SAVE_DIR, 'pdscnn_swa.pth')
HISTORY_PATH    = os.path.join(DRIVE_SAVE_DIR, 'training_history.csv')
CONFIG_PATH     = os.path.join(DRIVE_SAVE_DIR, 'config.json')

EPOCHS          = 100
LR              = 1e-3    # OneCycleLR handles the warm-up so start here
WEIGHT_DECAY    = 1e-4
LABEL_SMOOTHING = 0.10
MIXUP_ALPHA     = 0.4
SWA_START       = 75     # start SWA after epoch 75

optimizer = torch.optim.AdamW(pdscnn.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=SWA_START,
    pct_start=0.10,
    anneal_strategy='cos',
    div_factor=25,
    final_div_factor=1e4,
)
criterion = nn.CrossEntropyLoss(weight=class_weights_t, label_smoothing=LABEL_SMOOTHING)
scaler    = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

# SWA for final accuracy boost
swa_model     = swa_utils.AveragedModel(pdscnn)
swa_scheduler = swa_utils.SWALR(optimizer, swa_lr=5e-5, anneal_epochs=5)

history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
best_val_acc = 0.0

for epoch in range(EPOCHS):
    pdscnn.train()
    running_loss, running_correct, n_seen = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]")
    use_scheduler_step = epoch < SWA_START

    for imgs, labels in pbar:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        # MixUp augmentation
        mixed_imgs, y_a, y_b, lam = mixup_data(imgs, labels, alpha=MIXUP_ALPHA)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits, _ = pdscnn(mixed_imgs)
            loss = mixup_criterion(criterion, logits, y_a, y_b, lam)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(pdscnn.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        if use_scheduler_step:
            scheduler.step()

        running_loss += loss.item() * imgs.size(0)
        # Approximate train acc via MixUp-weighted prediction (avoids a 2nd forward pass)
        with torch.no_grad():
            pred = logits.detach().argmax(1)
            correct = lam * (pred == y_a).float().sum() + (1 - lam) * (pred == y_b).float().sum()
            running_correct += correct.item()
        n_seen += imgs.size(0)
        pbar.set_postfix(loss=running_loss / n_seen, acc=running_correct / n_seen)

    train_loss = running_loss / n_seen
    train_acc  = running_correct / n_seen

    # SWA update
    if epoch >= SWA_START:
        swa_model.update_parameters(pdscnn)
        swa_scheduler.step()

    # Validation (no MixUp, no gradient)
    pdscnn.eval()
    val_loss, val_correct, n_val = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                logits, _ = pdscnn(imgs)
                loss = criterion(logits, labels)
            val_loss    += loss.item() * imgs.size(0)
            val_correct += (logits.argmax(1) == labels).sum().item()
            n_val       += imgs.size(0)

    val_loss /= n_val
    val_acc   = val_correct / n_val
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss {train_loss:.4f} train_acc {train_acc:.4f} "
          f"| val_loss {val_loss:.4f} val_acc {val_acc:.4f} | lr {current_lr:.2e}")

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': pdscnn.state_dict(),
            'epoch': epoch + 1,
            'val_acc': val_acc,
            'class_names': CLASS_NAMES,
            'dataset_mean': DATASET_MEAN,
            'dataset_std': DATASET_STD,
            'img_size': IMG_SIZE,
        }, BEST_CKPT_PATH)
        print(f"  -> New best val_acc {val_acc:.4f}, checkpoint saved!")

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

# Finalise SWA: update BatchNorm stats then save
print("Updating SWA BatchNorm stats...")
torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)
torch.save(swa_model.state_dict(), SWA_CKPT_PATH)
print(f"SWA model saved to {SWA_CKPT_PATH}")

torch.save({
    'model_state_dict': pdscnn.state_dict(),
    'epoch': EPOCHS,
    'val_acc': val_acc,
    'class_names': CLASS_NAMES,
    'dataset_mean': DATASET_MEAN,
    'dataset_std': DATASET_STD,
    'img_size': IMG_SIZE,
}, FINAL_CKPT_PATH)

with open(CONFIG_PATH, 'w') as cfg_f:
    json.dump({
        'class_names': CLASS_NAMES,
        'dataset_mean': DATASET_MEAN,
        'dataset_std': DATASET_STD,
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'label_smoothing': LABEL_SMOOTHING,
        'mixup_alpha': MIXUP_ALPHA,
        'swa_start': SWA_START,
        'best_val_acc': best_val_acc,
        'num_train': len(train_df), 'num_val': len(val_df), 'num_test': len(test_df),
    }, cfg_f, indent=2)

print(f"\nTraining complete. Best val_acc: {best_val_acc:.4f}")
print(f"All outputs saved to: {DRIVE_SAVE_DIR}")


## 9. Training Curves

In [ ]:
hist_df = pd.DataFrame(history)

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].plot(hist_df['epoch'], hist_df['train_loss'], label='train')
axs[0].plot(hist_df['epoch'], hist_df['val_loss'], label='val')
axs[0].set_title('Loss'); axs[0].set_xlabel('Epoch'); axs[0].legend(); axs[0].grid(alpha=0.3)

axs[1].plot(hist_df['epoch'], hist_df['train_acc'], label='train')
axs[1].plot(hist_df['epoch'], hist_df['val_acc'], label='val')
axs[1].set_title('Accuracy'); axs[1].set_xlabel('Epoch'); axs[1].legend(); axs[1].grid(alpha=0.3)

axs[2].plot(hist_df['epoch'], hist_df['lr'])
axs[2].set_title('Learning rate'); axs[2].set_xlabel('Epoch'); axs[2].grid(alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(DRIVE_SAVE_DIR, 'training_curves.png')
plt.savefig(curves_path, dpi=150)
plt.show()
print("Saved:", curves_path)


## 10. Test Set Evaluation

In [ ]:
ckpt = torch.load(BEST_CKPT_PATH, map_location=device)
pdscnn.load_state_dict(ckpt['model_state_dict'])
pdscnn.eval()
print(f"Loaded best checkpoint from epoch {ckpt['epoch']} (val_acc {ckpt['val_acc']:.4f})")

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Evaluating on test set"):
        imgs = imgs.to(device)
        logits, _ = pdscnn(imgs)
        probs = F.softmax(logits, dim=1)
        all_preds.extend(probs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')

print(f"\nTest Accuracy: {test_acc:.4f}")
print(f"Macro Precision: {precision:.4f} | Macro Recall: {recall:.4f} | Macro F1: {f1:.4f}\n")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('PDSCNN — Test Confusion Matrix')

for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.colorbar(im, ax=ax)
plt.tight_layout()
cm_path = os.path.join(DRIVE_SAVE_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print("Saved:", cm_path)


## 11. Wrap-up

Everything needed to reuse this model later (e.g. as the PDSCNN branch of the hybrid fusion pipeline) is saved in **`/content/drive/MyDrive/pdscnn_standalone/`**:

- `pdscnn_best.pth` — best checkpoint (state dict + class names + normalization stats + img size)
- `pdscnn_final.pth` — checkpoint after all 50 epochs
- `training_history.csv` — per-epoch loss/accuracy/lr log
- `training_curves.png`, `confusion_matrix.png`
- `config.json` — full run configuration for reproducibility

To reload the model later:

```python
ckpt = torch.load('/content/drive/MyDrive/pdscnn_standalone/pdscnn_best.pth', map_location=device)
model = PDSCNN(num_classes=len(ckpt['class_names'])).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
```


## 12. RRELM Feature Extraction and Classification

In [ ]:

from sklearn.preprocessing import StandardScaler

def extract_features(model, loader):
    model.eval()
    all_feats, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Extracting"):
            imgs = imgs.to(device)
            _, feats = model(imgs)
            all_feats.append(feats.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_feats), np.concatenate(all_labels)

# We use the train_loader but without shuffle/augmentations for extraction
train_loader_eval = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

X_train, y_train = extract_features(pdscnn, train_loader_eval)
X_test, y_test = extract_features(pdscnn, test_loader)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

class RRELM:
    def __init__(self, n_hidden=1500, C=10.0, activation='sigmoid', random_state=42):
        self.n_hidden = n_hidden
        self.C = C
        self.activation = activation
        self.rng = np.random.RandomState(random_state)

    def _activate(self, H):
        return 1.0 / (1.0 + np.exp(-H))

    def fit(self, X, y_onehot):
        self.W = self.rng.uniform(-1, 1, size=(X.shape[1], self.n_hidden))
        self.b = self.rng.uniform(-1, 1, size=(self.n_hidden,))
        H = self._activate(X @ self.W + self.b)
        n = H.shape[0]
        if n >= self.n_hidden:
            I = np.eye(self.n_hidden)
            self.beta = np.linalg.solve(H.T @ H + I / self.C, H.T @ y_onehot)
        else:
            I = np.eye(n)
            self.beta = H.T @ np.linalg.solve(H @ H.T + I / self.C, y_onehot)
        return self

    def predict_proba(self, X):
        H = self._activate(X @ self.W + self.b)
        return H @ self.beta

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

def to_onehot(y, n_classes):
    oh = np.zeros((len(y), n_classes))
    oh[np.arange(len(y)), y] = 1
    return oh

rrelm = RRELM(n_hidden=1500, C=10.0, random_state=SEED)
rrelm.fit(X_train_s, to_onehot(y_train, 4))

test_preds = rrelm.predict(X_test_s)
acc = accuracy_score(y_test, test_preds)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, test_preds, average='macro')
print(f"RRELM Test Accuracy: {acc:.4f}")
print(f"Macro Precision: {precision:.4f} | Macro Recall: {recall:.4f} | Macro F1: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=CLASS_NAMES))
